# 分块边界：按页还是按句子

固定长度分块可能从一句话中间切开，也可能把同一页中的几项说明拆散。下面保留两种改法，但分别用适合它们的问题验证：按页分块保住同页说明，按句子边界分块保住完整句子。

每次比较都从同一份南瓜书 PDF 读取资料，只改变切分边界。检索仍用 BM25；上下文预算用字符数明确记下，避免把更长的整页误当成同等成本。

In [1]:
import json
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
        nested = folder / "notebook" / "C7 高级 RAG 技巧"
        if (nested / "data" / "dataset/manifest.json").is_file():
            return nested
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

In [2]:
from common.eval_utils import (
    build_bm25_chunk_search,
    load_query_catalog,
    load_pdf_pages,
    make_fixed_chunks,
    make_recursive_chunks,
)

cases = {item["id"]: item for item in load_query_catalog()}
page_case = cases["descent_direction"]
required_parts = [
    ("梯度指向的方向是函数值增大速度最快的方向", "梯度的反方向"),
    ("xt+1", "步长", "学习率"),
]
pages = load_pdf_pages()
page_fixed_size = 300
page_fixed_chunks = make_fixed_chunks(pages, chunk_size=page_fixed_size, overlap=0)
page_chunks = [
    {"chunk_id": f"p{page['page']}_full", "pages": [page["page"]], "text": page["text"]}
    for page in pages
]

print("问题：", page_case["query"])
print("需要同时保留：下降方向的解释；更新式和步长含义")
print("资料来源：同一份南瓜书 PDF；页面总数：", len(pages))
print("固定长度片段数：", len(page_fixed_chunks))
print("按页片段数：", len(page_chunks))

问题： 在优化过程中，如何通过调整方向来确保函数值逐步减小？
需要同时保留：下降方向的解释；更新式和步长含义
资料来源：同一份南瓜书 PDF；页面总数： 196
固定长度片段数： 1050
按页片段数： 196


## 按页分块：先把预算说清楚

问题是“在优化过程中，如何通过调整方向来确保函数值逐步减小？”。它需要同时看到两项：为什么朝梯度反方向走，以及更新式里的步长含义。按页的片段天然更长，所以这里固定一个 1700 字符的上下文预算：固定长度切分最多取 5 个 300 字符片段，按页切分取 1 页。两边都在同一预算内，再比较这两项内容是否找全。

下面再用问题集中的 PCA 投影问题做一个短检查：两种切法最终都只返回前 5 条，并打印实际字符数。目标排名是另外查询前 20 条结果得到的诊断排名，不改变最终返回的前 5 条。按页结果会明显更长，所以这道题只能说明多保留了哪些内容，不能单独证明页面边界比固定长度更好。

In [3]:
from common.eval_utils import emit_tutorial_audit
from common.nontraining_utils import load_annotation

context_budget = 1700
fixed_k = context_budget // page_fixed_size
fixed_results = build_bm25_chunk_search(page_fixed_chunks)(page_case["query"], top_k=fixed_k)
page_results = build_bm25_chunk_search(page_chunks)(page_case["query"], top_k=1)
page_annotation = load_annotation(page_case["id"])

def answer_part_coverage(results):
    context = " ".join(item.text for item in results)
    return sum(all(text in context for text in part) for part in required_parts) / len(required_parts)

def result_metrics(results, expected_pages):
    pages = [int(page) for item in results for page in item.pages]
    expected = set(int(page) for page in expected_pages)
    found = expected.intersection(pages)
    return {
        'pages': pages,
        'first_required_rank': next((index for index, item in enumerate(results, 1) if expected.intersection(item.pages)), None),
        'required_page_coverage': len(found) / len(expected) if expected else 0.0,
    }

before_chars = sum(len(item.text) for item in fixed_results)
after_chars = sum(len(item.text) for item in page_results)
before_coverage = answer_part_coverage(fixed_results)
after_coverage = answer_part_coverage(page_results)
print("固定长度取前", len(fixed_results), "条，页面：", sorted({page for item in fixed_results for page in item.pages}))
print("按页取", len(page_results), "条，页面：", sorted({page for item in page_results for page in item.pages}))
print("上下文字符数（预算", context_budget, "）：", before_chars, "→", after_chars)
print("两部分回答的覆盖率：", before_coverage, "→", after_coverage)
print("复核：固定长度是否漏掉至少一项：", before_coverage < 1.0, "；按页是否找全：", after_coverage == 1.0)

assert before_chars <= context_budget and after_chars <= context_budget and (before_coverage, after_coverage) == (0.5, 1.0)
emit_tutorial_audit({
    'case_id': 'descent_direction',
    'method': '按页分块',
    'role': 'main',
    'before': result_metrics(fixed_results, page_annotation['expected_pages']),
    'after': result_metrics(page_results, page_annotation['expected_pages']),
})

pca_case = cases["pca_projection_objective"]
pca_compare_k = 5
pca_compare_budget = 8000
pca_fixed_chunks = make_fixed_chunks(pages, chunk_size=260, overlap=0)
pca_fixed_search = build_bm25_chunk_search(pca_fixed_chunks)
pca_page_search = build_bm25_chunk_search(page_chunks)
pca_fixed_top = pca_fixed_search(pca_case["query"], top_k=pca_compare_k)
pca_page_top = pca_page_search(pca_case["query"], top_k=pca_compare_k)
# 另查前 20 条只用于诊断目标排名；最终上下文仍只返回前 5 条。
pca_fixed_ranked = pca_fixed_search(pca_case["query"], top_k=20)
pca_page_ranked = pca_page_search(pca_case["query"], top_k=20)
pca_annotation = load_annotation(pca_case["id"])
pca_target_pages = set(pca_annotation["expected_pages"])

def first_target_rank(results, target_pages):
    return next((rank for rank, item in enumerate(results, 1) if target_pages.intersection(item.pages)), None)

def keyword_coverage(results, keywords):
    context = " ".join(item.text for item in results)
    return sum(keyword in context for keyword in keywords), len(keywords)

pca_fixed_chars = sum(len(item.text) for item in pca_fixed_top)
pca_page_chars = sum(len(item.text) for item in pca_page_top)
pca_fixed_rank = first_target_rank(pca_fixed_ranked, pca_target_pages)
pca_page_rank = first_target_rank(pca_page_ranked, pca_target_pages)
pca_fixed_coverage = keyword_coverage(pca_fixed_top, pca_annotation["expected_keywords"])
pca_page_coverage = keyword_coverage(pca_page_top, pca_annotation["expected_keywords"])
print("\n按页复核问题：", pca_case["query"])
print("改前（固定长度，最终返回前 5 条）页码：", [item.pages[0] for item in pca_fixed_top], "；目标排名（另查前 20 条，仅作诊断）：", pca_fixed_rank or "前 20 条没有找到")
print("改后（按页，最终返回前 5 条）页码：", [item.pages[0] for item in pca_page_top], "；目标排名（另查前 20 条，仅作诊断）：", pca_page_rank or "前 20 条没有找到")
print("返回资料量（最终只返回前 5 条）：", len(pca_fixed_top), "→", len(pca_page_top), "；上下文字符数（上限", pca_compare_budget, "）：", pca_fixed_chars, "→", pca_page_chars)
print("证据词覆盖（W、zi、Z、目标函数）：", pca_fixed_coverage, "→", pca_page_coverage)
print("结论：按页结果找回目标页并覆盖全部证据词，但使用的文字明显更多；这道题只记录收益与代价，不作为同等资料量下的改善证明。")
assert pca_fixed_chars <= pca_compare_budget and pca_page_chars <= pca_compare_budget and pca_fixed_rank > pca_compare_k >= pca_page_rank and pca_fixed_coverage[0] < pca_page_coverage[0]
emit_tutorial_audit({
    'case_id': 'pca_projection_objective',
    'method': '按页分块',
    'role': 'check',
    'before': result_metrics(pca_fixed_top, pca_annotation['expected_pages']),
    'after': result_metrics(pca_page_top, pca_annotation['expected_pages']),
    'check_purpose': '说明不适用或限制',
})


固定长度取前 5 条，页面： [38, 96, 106, 140, 142]
按页取 1 条，页面： [38]
上下文字符数（预算 1700 ）： 1500 → 1567
两部分回答的覆盖率： 0.5 → 1.0
复核：固定长度是否漏掉至少一项： True ；按页是否找全： True




按页复核问题： PCA 降维时低维坐标如何由 W 得到，W 的优化目标是什么？
改前（固定长度，最终返回前 5 条）页码： [126, 133, 130, 126, 41] ；目标排名（另查前 20 条，仅作诊断）： 12
改后（按页，最终返回前 5 条）页码： [126, 133, 130, 131, 127] ；目标排名（另查前 20 条，仅作诊断）： 5
返回资料量（最终只返回前 5 条）： 5 → 5 ；上下文字符数（上限 8000 ）： 1299 → 7057
证据词覆盖（W、zi、Z、目标函数）： (3, 4) → (4, 4)
结论：按页结果找回目标页并覆盖全部证据词，但使用的文字明显更多；这道题只记录收益与代价，不作为同等资料量下的改善证明。



## 主要问题：按句子边界保住连续属性的完整规则

连续属性问题要解释 16 个候选划分点怎样得到。两种切法都来自同一份 PDF，都把片段上限设为 160 字、不重叠，并让前 5 条结果进入同样的 800 字符预算。固定长度切法漏掉了“相邻取值的中点”这项说明；按句子边界切分后，两项说明都进入前 5 条资料。

再用 LASSO 稀疏解做限制检查：只检查问题集真正需要的“稀疏、非零分量个数尽量小”两项证据，不把“过拟合”或题名本身当作答案必要项。若固定长度已经找全，这个复核就只说明该题原本已足够，不把没有改善写成收益。

In [4]:
from common.eval_utils import emit_tutorial_audit

case = cases["continuous_attribute_split_points"]
sentence_chunk_size = 160
sentence_fixed_chunks = make_fixed_chunks(pages, chunk_size=sentence_chunk_size, overlap=0)
sentence_chunks = make_recursive_chunks(pages, chunk_size=sentence_chunk_size, overlap=0)
required_parts = [
    ("相邻取值的中点",),
    ("i 依次取1 到16", "候选划分点集合"),
]

sentence_result_k = 5
sentence_context_budget = sentence_result_k * sentence_chunk_size
fixed_results = build_bm25_chunk_search(sentence_fixed_chunks)(case["query"], top_k=sentence_result_k)
sentence_results = build_bm25_chunk_search(sentence_chunks)(case["query"], top_k=sentence_result_k)
sentence_annotation = load_annotation(case["id"])

def answer_part_coverage(results):
    context = " ".join(item.text for item in results)
    found = sum(all(text in context for text in part) for part in required_parts)
    return found / len(required_parts)


before_coverage = answer_part_coverage(fixed_results)
after_coverage = answer_part_coverage(sentence_results)

print("主要问题：", case["query"])
print("两种切法的最长片段：", max(len(chunk["text"]) for chunk in sentence_fixed_chunks), "和", max(len(chunk["text"]) for chunk in sentence_chunks), "字")
sentence_fixed_chars = sum(len(item.text) for item in fixed_results)
sentence_chars = sum(len(item.text) for item in sentence_results)
print("返回资料量：", len(fixed_results), "→", len(sentence_results), "；字符预算：", sentence_context_budget)
print("前 5 条上下文字符数：", sentence_fixed_chars, "→", sentence_chars)
print("固定长度，前 5 条所在页：", [item.pages[0] for item in fixed_results])
print("按句子边界，前 5 条所在页：", [item.pages[0] for item in sentence_results])
print("两部分回答的覆盖率：", before_coverage, "→", after_coverage)
print("复核：按句子边界的前 5 条是否包含两项：", after_coverage == 1.0)

assert len(fixed_results) == len(sentence_results) == sentence_result_k and sentence_fixed_chars <= sentence_context_budget and sentence_chars <= sentence_context_budget and max(len(chunk["text"]) for chunk in sentence_fixed_chunks) <= sentence_chunk_size and max(len(chunk["text"]) for chunk in sentence_chunks) <= sentence_chunk_size and (before_coverage, after_coverage) == (0.5, 1.0)
emit_tutorial_audit({
    'case_id': 'continuous_attribute_split_points',
    'method': '按句子边界分块',
    'role': 'main',
    'before': result_metrics(fixed_results, sentence_annotation['expected_pages']),
    'after': result_metrics(sentence_results, sentence_annotation['expected_pages']),
})

lasso_case = cases["lasso_sparse_solution"]
lasso_fixed_chunks = make_fixed_chunks(pages, chunk_size=260, overlap=0)
lasso_sentence_chunks = make_recursive_chunks(pages, chunk_size=260, overlap=0)
lasso_fixed_search = build_bm25_chunk_search(lasso_fixed_chunks)
lasso_sentence_search = build_bm25_chunk_search(lasso_sentence_chunks)
lasso_result_k = 5
lasso_context_budget = lasso_result_k * 260
lasso_fixed_top = lasso_fixed_search(lasso_case["query"], top_k=lasso_result_k)
lasso_sentence_top = lasso_sentence_search(lasso_case["query"], top_k=lasso_result_k)
lasso_fixed_ranked = lasso_fixed_search(lasso_case["query"], top_k=20)
lasso_sentence_ranked = lasso_sentence_search(lasso_case["query"], top_k=20)
lasso_annotation = load_annotation(lasso_case["id"])
lasso_target_pages = set(lasso_annotation["expected_pages"])
lasso_evidence = [("稀疏",), ("非零分量个数尽量小",)]

def lasso_evidence_coverage(results):
    context = " ".join(item.text for item in results)
    return sum(all(text in context for text in part) for part in lasso_evidence), len(lasso_evidence)

lasso_fixed_rank = first_target_rank(lasso_fixed_ranked, lasso_target_pages)
lasso_sentence_rank = first_target_rank(lasso_sentence_ranked, lasso_target_pages)
lasso_fixed_coverage = lasso_evidence_coverage(lasso_fixed_top)
lasso_sentence_coverage = lasso_evidence_coverage(lasso_sentence_top)
lasso_fixed_chars = sum(len(item.text) for item in lasso_fixed_top)
lasso_sentence_chars = sum(len(item.text) for item in lasso_sentence_top)
print("\n限制检查（LASSO 原本已足够）：", lasso_case["query"])
print("改前（固定长度，前 5 条）页码：", [item.pages[0] for item in lasso_fixed_top], "；目标排名：", lasso_fixed_rank or "前 20 条没有找到")
print("改后（按句子边界，前 5 条）页码：", [item.pages[0] for item in lasso_sentence_top], "；目标排名：", lasso_sentence_rank or "前 20 条没有找到")
print("返回资料量：", len(lasso_fixed_top), "→", len(lasso_sentence_top), "；字符预算：", lasso_context_budget)
print("上下文字符数：", lasso_fixed_chars, "→", lasso_sentence_chars, "；检索调用：2 → 2")
print("必要回答证据覆盖（稀疏、非零分量个数尽量小）：", lasso_fixed_coverage, "→", lasso_sentence_coverage)
print("限制结论：LASSO 的必要回答已在固定长度首条资料中找全；按句子边界没有增加覆盖。")
assert len(lasso_fixed_top) == len(lasso_sentence_top) == lasso_result_k and lasso_fixed_chars <= lasso_context_budget and lasso_sentence_chars <= lasso_context_budget and lasso_fixed_coverage == lasso_sentence_coverage == (2, 2)
emit_tutorial_audit({
    'case_id': 'lasso_sparse_solution',
    'method': '按句子边界分块',
    'role': 'check',
    'before': result_metrics(lasso_fixed_top, lasso_annotation['expected_pages']),
    'after': result_metrics(lasso_sentence_top, lasso_annotation['expected_pages']),
    'check_purpose': '说明不适用或限制',
})


主要问题： 南瓜书的‘密度’属性有17个已观测取值，候选集合 T_a 为什么有16个划分点？每个划分点怎样计算？
两种切法的最长片段： 160 和 160 字
返回资料量： 5 → 5 ；字符预算： 800
前 5 条上下文字符数： 800 → 798
固定长度，前 5 条所在页： [49, 52, 49, 45, 45]
按句子边界，前 5 条所在页： [52, 45, 51, 49, 49]
两部分回答的覆盖率： 0.5 → 1.0
复核：按句子边界的前 5 条是否包含两项： True


限制检查（LASSO 原本已足够）： LASSO 的 L1 范数为什么更容易得到稀疏解？
改前（固定长度，前 5 条）页码： [140, 143, 143, 147, 147] ；目标排名： 1
改后（按句子边界，前 5 条）页码： [140, 143, 147, 143, 147] ；目标排名： 1
返回资料量： 5 → 5 ；字符预算： 1300
上下文字符数： 1182 → 1019 ；检索调用：2 → 2
必要回答证据覆盖（稀疏、非零分量个数尽量小）： (2, 2) → (2, 2)
限制结论：LASSO 的必要回答已在固定长度首条资料中找全；按句子边界没有增加覆盖。



## 结论

梯度下降问题中，在约 1700 字符的同一上下文预算内，固定长度切分取了 5 条片段却只找全一项；按页切分取 1 页并找全两项。按页片段的确更长，但这里把实际字符数打印出来了，结论是“在这份 PDF、这个预算和这个问题上，页面边界保住了跨段说明”，不代表按页在所有预算下都更好。

在另一道题上，两种切法保持相同长度上限、相同返回数量和相同字符预算，按句子边界分块把两部分回答的覆盖率从 0.5 提高到 1.0。PCA 检查中目标排名和证据覆盖虽然改善，但文字量从 1299 增至 7057，因此这里只记录代价，不把它算作公平比较；其中目标排名来自另查前 20 条的诊断，最终仍只返回前 5 条。LASSO 限制检查只看稀疏性与非零分量两个必要要点，覆盖率保持 2/2 → 2/2，说明该题原本已足够。页面很长、主题混杂时不要盲目按页保存；一句话没有被截断时，也不必为了句子边界重切全部资料。

## 从“能切开”到“能回答”

调整分块不是为了让片段数量变多，而是让每个片段尽量表达一个完整意思。一个片段包含多个主题时，生成的向量可能无法准确代表任何一个主题；一句话从中间切开时，必要条件和结论又可能落在不同片段。整页或超长片段虽然保住了上下文，却会带来更多无关文字、token 消耗和误引用。

本页保留了两种有明确边界的改法。梯度下降问题需要同时看到“沿梯度反方向”和“更新式中的步长”，所以按页切分在约 1700 字符预算内找全了两项；这只说明页面边界适合这个问题，不代表所有 PDF 都应按页。连续属性问题需要保住“相邻取值的中点”和候选划分点的生成规则，按句子边界在相同前 5 条、160 字上限下提高了必要证据覆盖。PCA 对照还显示按页或句子边界可能增加很多文字，LASSO 对照则显示原切分已经足够，因此不能把每次变化都称作改善。

评估时至少同时看三件事：目标页或目标片段的排名、答案必要要点是否完整、最终交给回答模型的字符/token 数。只有第一项变化而第二项没有变化时，应该把它记录为排序变化，而不是回答质量提升。


## 证据覆盖计算的代码写法

下面只展示证据覆盖计算的代码写法，不产生本页的比较结果；正式比较仍使用上面的固定输入。

```python
def evidence_coverage(results, required_phrases):
    context = " ".join(item.text for item in results)
    return sum(
        all(phrase in context for phrase in alternatives)
        for alternatives in required_phrases
    ) / len(required_phrases)

# required_phrases 的每个元素可以是一个同义词集合：
# [(“相邻取值的中点”,), (“候选划分点集合”, “i 依次取1 到16”)]
# 这样评估的是回答真正需要的证据，而不是题名或偶然出现的词。
```
